In [2]:
import glob 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d


In [ ]:
# df = pd.read_csv('./for_gijs.csv')
# ctl = pd.read_csv('./take_two_stellar_params_CTL.csv')

# # Normalize TIC identifiers before merging.
# # Some values may include prefixes, decimal suffixes, or stray whitespace.
# df['TIC'] = df['TIC'].astype('str').str.extract(r'(\d+)', expand=False)
# ctl['id'] = ctl['id'].astype('str').str.extract(r'(\d+)', expand=False)

# df['TIC'] = df['TIC'].astype('int')
# ctl['id'] = ctl['id'].astype('int')

# # print(len(df), len(ctl))
# # Drop rows that still do not have a valid identifier.
# df = df.dropna(subset=['TIC'])
# ctl = ctl.dropna(subset=['id'])

# # print(ctl['id'].head())
# # print(df['TIC'].head())


# merged = df.merge(
#     ctl,
#     left_on='TIC',
#     right_on='id',
#     how='left',
#     validate='many_to_one',  # df can have repeats, ctl should not
#     indicator=True,
#  )


# merged['temp_bin'] = pd.cut(merged['Stellar Temperature'], bins=np.arange(3000, 10000, 500))
# merged['temp_bin_center'] = merged['temp_bin'].apply(lambda x: x.mid)
# print(merged.head(100))



In [4]:
df = pd.read_csv('./for_gijs.csv')

df['temp_bin'] = pd.cut(df['Stellar Temperature'], bins=np.arange(3000, 10000, 500))
df['temp_bin_center'] = df['temp_bin'].apply(lambda x: x.mid)

# Normalize TIC identifiers before merging.
# Some values may include prefixes, decimal suffixes, or stray whitespace.

data = {
    'Stellar Type': ['M0V', 'M1V', 'M2V', 'M3V', 'M4V', 'M5V', 'M6V', 'K0V', 'K1V', 'K2V', 'K3V', 'K4V', 'K5V', 'K6V', 'K7V', 'K8V', 'K9V',
                     'G0V', 'G1V', 'G2V', 'G3V', 'G4V', 'G5V', 'G6V', 'G7V', 'G8V', 'G9V',
                     'F0V', 'F1V', 'F2V', 'F3V', 'F4V', 'F5V', 'F6V', 'F7V', 'F8V', 'F9V'],
    'Mass': [0.57, 0.50, 0.44, 0.37, 0.23, 0.162, 0.102, 0.88, 0.86, 0.82, 0.78, 0.73, 0.70, 0.69, 0.64, 0.62, 0.59,
             1.06, 1.03, 1.00, 0.99, 0.985, 0.98, 0.97, 0.95, 0.94, 0.90,
             1.61, 1.50, 1.46, 1.44, 1.38, 1.33, 1.25, 1.21, 1.18, 1.13],
    'Radius': [0.588, 0.501, 0.446, 0.361, 0.274, 0.196, 0.137, 0.813, 0.797, 0.783, 0.755, 0.713, 0.701, 0.669, 0.630, 0.615, 0.608,
               1.100, 1.060, 1.012, 1.002, 0.991, 0.977, 0.949, 0.927, 0.914, 0.853,
               1.728, 1.679, 1.622, 1.578, 1.533, 1.473, 1.359, 1.324, 1.221, 1.167],
    'Luminosity': [0.069, 0.041, 0.029, 0.016, 7.2e-3, 3.0e-3, 1.0e-3, 0.46, 0.41, 0.37, 0.28, 0.20, 0.17, 0.14, 0.10, 0.087, 0.079,
                   1.35, 1.20, 1.02, 0.98, 0.91, 0.89, 0.79, 0.74, 0.68, 0.55,
                   7.24, 6.17, 5.13, 4.68, 4.17, 3.63, 2.69, 2.45, 1.95, 1.66],
    'Effective Temperature': [3850, 3660, 3560, 3430, 3210, 3060, 2810, 5270, 5170, 5100, 4830, 4600, 4440, 4300, 4100, 3990, 3930,
                              5930, 5860, 5770, 5720, 5680, 5660, 5600, 5550, 5480, 5380,
                              7220, 7020, 6820, 6750, 6670, 6550, 6350, 6280, 6180, 6050],
    'B-V Index': [1.42, 1.49, 1.51, 1.53, 1.65, 1.83, 2.01, 0.82, 0.86, 0.88, 0.99, 1.09, 1.15, 1.24, 1.34, 1.36, 1.40,
                  0.60, 0.62, 0.65, 0.66, 0.67, 0.68, 0.70, 0.71, 0.73, 0.78,
                  0.30, 0.33, 0.37, 0.39, 0.41, 0.44, 0.49, 0.50, 0.53, 0.56]
}



stellar_df = pd.DataFrame(data)

# Sort by temperature for interpolation
stellar_df = stellar_df.sort_values('Effective Temperature')

# Create Teff -> Mass interpolation function
mass_interp = interp1d(
    stellar_df['Effective Temperature'],
    stellar_df['Mass'],
    kind='linear',
    bounds_error=False,
    fill_value='extrapolate'
)

# Get mass from temperature bin center
df['Stellar Mass Estimate'] = mass_interp(df['temp_bin_center'])
print(df['Stellar Mass Estimate'].unique())

[0.7626087  0.876      0.996      0.6775     0.25545455 1.201
 1.44       0.53315789]


In [5]:
df['empirical_radius_limit'] = 10**(0.23 * np.log10(df['Stellar Mass Estimate']) + 0.27)

print(df['empirical_radius_limit'].min(), df['empirical_radius_limit'].max())

1.360448593516989 2.024992286131788


In [6]:
limits_df = pd.DataFrame({
    'Temperature': sorted(df['temp_bin_center'].dropna().unique()),
    'Radius': sorted(df['empirical_radius_limit'].dropna().unique())
})

print(limits_df)

   Temperature    Radius
0       3250.0  1.360449
1       3750.0  1.611302
2       4250.0  1.702586
3       4750.0  1.749562
4       5250.0  1.806242
5       5750.0  1.860371
6       6250.0  1.942204
7       6750.0  2.024992


In [11]:
df['empirical_radius_limit_from_period'] = 10**(-0.11 * np.log10(df['True Period (Days)']) + 0.37)

print(df['empirical_radius_limit_from_period'].min(), df['empirical_radius_limit_from_period'].max())
print(df['True Period (Days)'].min(), df['True Period (Days)'].max())

1.5922258704958328 2.519066498578457
0.52 33.669
